# EDA: NASA Meteorite Landings

Work through this top to bottom. Some code is given as a model; most of it
you write yourself this time — this module teaches real pandas **and**
Polars, not just reading given output. See `../SCENARIOS.md` to pick your
stakeholder/question first if you haven't already, `../README.md` for
setup, and `../MVP.md` for the full scope.


In [ ]:
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

df = pd.read_csv("../data/meteorite_landings.csv")
df.shape


## Part 1: Profiling with pandas

A **five-number summary** (min, 25th percentile, median, 75th percentile,
max) plus a **histogram** tells you a variable's shape — symmetric or
skewed — and whether it has real outliers. Look at the histogram's shape
first, *then* check whether the five-number summary agrees, not the other
way around.

### Worked example: `mass (g)`

This one's fully done for you, as a model for your own scenario's
variable(s) below — both the code and the written interpretation.


In [ ]:
print(df["mass (g)"].describe())
df["mass (g)"].plot(kind="hist", bins=40, title="mass (g)")
plt.show()


**Shape:** Heavily **right-skewed**. The mean (about 13,278 g) is roughly
400x the median (32.6 g) — that gap alone is a strong tell, since a
symmetric distribution's mean and median sit close together. Most
meteorites are small; a small number of very large ones drag the mean way
up.

**Outliers:** Yes — the max value is 60,000,000 g (60 metric tons, the Hoba
meteorite). Not a data error, just a genuinely enormous real object — but
it's exactly the kind of value that makes the mean misleading here: **the
median (32.6 g), not the mean, is the better answer to "how big is a
typical meteorite?"**


### A few pandas moves you'll likely need

Reference, not the analysis itself — pick whichever apply to your chosen
scenario's question:

- **Filter rows:** `df[df["fall"] == "Fell"]` (boolean masking — keeps
  only rows where the condition is `True`).
- **Group and summarize:** `df.groupby("recclass")["mass (g)"].describe()`
  — same five-number-summary idea, computed separately per group.
- **Sort:** `df.sort_values("mass (g)", ascending=False)` — biggest first.
- **A new column from an existing one:** `df["decade"] = (df["year"] // 10 * 10)`
  — integer division to bucket years into decades.
- **Top N by count:** `df["recclass"].value_counts().head(5)`.


### Your turn: explore your scenario's question

Using your chosen stakeholder's question from `SCENARIOS.md`, write real
pandas code below — filter, group, sort, whatever the question actually
needs. You'll likely want **at least one grouped summary** (like the
worked example, but split by a category) and **at least one sort or
filter** that isolates something specific. This is real, ungraded-template
exploration — expect to write more than one cell before you find something
worth reporting.


In [ ]:
# TODO: your exploration code for your chosen scenario's question

In [ ]:
# TODO: (add more cells as you need them)

**What did you find?** Summarize what your exploration above actually
shows — in plain language, tied to real numbers you can point to.

TODO


## Part 2: The Same Analysis in Polars

[Polars](https://pola.rs/) is a newer dataframe library — same job as
pandas, different engine (written in Rust, built for speed on larger data)
and a stricter, more explicit syntax. You're not replacing pandas — you're
learning to recognize the same operations in a second library, and reason
about when each one actually wins.

### pandas → Polars, side by side

| Operation | pandas | Polars |
|---|---|---|
| Read a CSV | `pd.read_csv(path)` | `pl.read_csv(path)` |
| Filter rows | `df[df["fall"] == "Fell"]` | `df.filter(pl.col("fall") == "Fell")` |
| Group + aggregate | `df.groupby("fall")["mass (g)"].agg(["count", "mean", "median"])` | `df.group_by("fall").agg([pl.len().alias("count"), pl.col("mass (g)").mean().alias("mean"), pl.col("mass (g)").median().alias("median")])` |
| Sort | `df.sort_values("mass (g)", ascending=False)` | `df.sort("mass (g)", descending=True)` |
| New column | `df["decade"] = df["year"] // 10 * 10` | `df.with_columns(((pl.col("year") // 10) * 10).alias("decade"))` |

Polars doesn't have a bare `.describe()`-per-group the way pandas does —
you build the exact summary columns you want inside `.agg([...])`, which
is more typing up front but makes precisely what's being computed explicit
(no implicit "whatever `.describe()` happens to include").

### Worked example: `mass (g)`, in Polars


In [ ]:
df_pl = pl.read_csv("../data/meteorite_landings.csv")
print(df_pl["mass (g)"].describe())


Same numbers as the pandas version above (median 32.6, heavily right-
skewed) — Polars' `.describe()` on a single Series works similarly to
pandas' here. The real syntax differences show up once you start
filtering/grouping (see the table above) — Polars wants you to name each
aggregation explicitly (`.agg([pl.col(...).mean().alias(...)])`) rather
than pandas' shorter `.agg(["mean"])`.


### Your turn: the same exploration, in Polars

Redo your Part 1 grouped summary (or whichever operation was most central
to your finding) using Polars this time — same question, same answer, new
syntax.


In [ ]:
# TODO: the Polars version of your Part 1 exploration

### A real timing comparison — run this cell twice

Given code — this times *this exact dataset's* size, so the result reflects
something real about pandas vs. Polars *at this scale*, not a claim from a
blog post. **Run this cell, then run it again without changing anything.**
Watch what happens to Polars' number the second time.


In [ ]:
import time

t0 = time.perf_counter()
_ = pd.read_csv("../data/meteorite_landings.csv").groupby("fall")["mass (g)"].agg(["count", "mean", "median"])
pandas_time = time.perf_counter() - t0

t0 = time.perf_counter()
_ = pl.read_csv("../data/meteorite_landings.csv").group_by("fall").agg([
    pl.len().alias("count"), pl.col("mass (g)").mean().alias("mean"), pl.col("mass (g)").median().alias("median")
])
polars_time = time.perf_counter() - t0

print(f"pandas: {pandas_time:.4f}s")
print(f"polars: {polars_time:.4f}s")


**Tradeoffs reflection.** Look at the real timing numbers you just got —
**both** runs, not just the first — and think back on the syntax
differences you hit while doing Part 2. Answer all four:

1. What changed about Polars' time between run 1 and run 2? (This is a
   real, documented Polars behavior, not a fluke — worth naming, not just
   noticing.)
2. Which one was actually faster **for this dataset, right now**? Was that
   what you expected, before you ran it?
3. Name one real syntax difference you personally ran into (not just from
   the cheat-sheet table — something you actually had to debug or look up).
4. Polars is generally built for **larger** data and **lazy** query
   chains (see `above_and_beyond/` if you want to test this for real). At
   this dataset's actual size, does "Polars is faster" hold up? Why might
   a real analyst still choose one over the other even when it's not the
   faster one *today*?

TODO


## Part 3: Get an AI's Read on Your Own Chart

Generate a chart from your own exploration, save it as a real image file,
then show that image to an AI assistant that can read images (the same
Claude/ChatGPT-style tool you already use) and ask it to describe what the
chart shows. This is different from Module 1's version of this exercise —
there, you were checking an AI's *text* description of a chart you didn't
make. Here, the AI is looking at a real image of **your own** work.


In [ ]:
# TODO: replace this with whichever variable/grouping is most central to
# your scenario's question -- this is given as a *pattern* to follow, not
# the specific chart you need.
fig, ax = plt.subplots()
df["mass (g)"].plot(kind="hist", bins=40, ax=ax, title="mass (g)")
fig.savefig("my_chart.png", dpi=150, bbox_inches="tight")
plt.show()


**Steps:**
1. Run the cell above (edited for your own variable/grouping) — confirm
   `my_chart.png` was actually created in this folder.
2. Open an AI assistant that accepts image uploads. Upload `my_chart.png`
   and ask it something like: *"What does this chart show about the
   distribution of this variable? Is it symmetric or skewed? Are there
   outliers?"*
3. Paste the AI's **real, actual response** below — not a paraphrase.


**The AI's response (pasted, verbatim):**

TODO


**Your critique.** Does the AI's read match what you already know from
your own five-number summary? Where does it agree? Where — if anywhere —
does it get something wrong, vague, or overconfident? An AI describing an
image has no access to your actual numbers unless you tell it — does its
answer show that limitation anywhere?

TODO


## Part 4: Answer Your Stakeholder's Question

Pull Parts 1-3 together into a direct answer to your chosen stakeholder's
actual question from `SCENARIOS.md`. State your answer plainly, back it
with the specific statistic (mean or median) that fits your variable's
actual skew, and say why that statistic — not the other one — is the
honest choice here.

TODO


## Part 5: Why does spread matter?

`mass (g)`'s standard deviation is about **574,989** — far larger than its
mean of 13,278. In your own words: what does a standard deviation that
much larger than the mean tell you about a real-world variable like this?
If this were a business metric instead of meteorite mass (e.g. monthly
sales, delivery times, insurance claim amounts), what would a standard
deviation that large imply about volatility, and why would that matter to
a stakeholder deciding whether to trust the mean as a planning number?

TODO


## Wrap-up

Once every section above is filled in: open `../starter/memo.md` and write
your findings report for your chosen stakeholder, plus the required
data-quality questions. Then check your work against `../MVP.md` before
your final commit.
